## Model Development

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import os

from model import TabTransformer, ImprovedTabTransformer

## 1. Load data & Preprocessing

In [52]:
# --- 1. Load Data ---
## df = pd.read_excel("../data/data_demo.xlsx")
df = pd.read_excel("../data/data_final.xlsx")

In [53]:
df.columns

Index(['url', 'filename', 'first_partner_gender', 'first_partner_age_bin',
       'second_partner_gender', 'second_partner_age_bin',
       'first_partner_school_category', 'second_partner_school_category',
       'first_partner_level_id', 'second_partner_level_id',
       'first_partner_field', 'second_partner_field'],
      dtype='object')

## Encode categorical features

In [54]:
# Clean missing data
df['first_partner_age_bin'] = df['first_partner_age_bin'].replace('Not mentioned', 'Unknown')
df['second_partner_age_bin'] = df['second_partner_age_bin'].replace('Not mentioned', 'Unknown')

label_encoders = {}
columns_to_encode = [
    'first_partner_gender', 'first_partner_age_bin', 'first_partner_school_category',
    'second_partner_gender', 'second_partner_age_bin', 'second_partner_school_category',
    'first_partner_level_id', 'second_partner_level_id',
    'first_partner_field', 'second_partner_field'
]

for col in columns_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))  # treat NaNs safely as "Unknown"
    label_encoders[col] = le

# Prepare input and output tensors 
input_cols = ['first_partner_gender', 'first_partner_age_bin', 'first_partner_school_category',
              'first_partner_level_id', 'first_partner_field']
output_cols = ['second_partner_gender', 'second_partner_age_bin', 'second_partner_school_category',
               'second_partner_level_id', 'second_partner_field']

## Split Data

In [55]:
# Split data
X_train, X_test, Y_train, Y_test = train_test_split(
    df[input_cols].values,
    df[output_cols].values,
    test_size=0.2,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
Y_train = torch.tensor(Y_train, dtype=torch.long)
Y_test = torch.tensor(Y_test, dtype=torch.long)

## Train and test datasets

In [56]:
# initialize 
input_cardinality = [df[col].nunique() for col in input_cols]
output_cardinality = [df[col].nunique() for col in output_cols]

# model
model = ImprovedTabTransformer(
    category_sizes=input_cardinality,
    dim=64,  
    output_sizes=output_cardinality
)



In [57]:
# Train the model
from torch.optim import AdamW
criterions = [nn.CrossEntropyLoss() for _ in output_cols]
# optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
optimizer = optim.AdamW(model.parameters(), lr=0.001)


def augment_batch(batch_X, mask_prob=0.2):
    mask = torch.rand_like(batch_X.float()) < mask_prob
    # Replace with "unknown" token (assuming 0 is unused/unknown)
    return torch.where(mask, torch.zeros_like(batch_X), batch_X)


def train(model, X, Y, epochs=15, batch_size=64):
    model.train()
    for epoch in range(epochs):
        permutation = torch.randperm(X.size()[0])
        total_loss = 0

        for i in range(0, X.size()[0], batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_Y = X[indices], Y[indices]
            batch_X = augment_batch(batch_X)

            optimizer.zero_grad()
            outputs = model(batch_X)

            losses = [
                criterions[j](outputs[j], batch_Y[:, j])
                for j in range(len(output_cols))
            ]
            loss = sum(losses)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f'Epoch {epoch+1}, Loss: {total_loss / len(X)}')
        
train(model, X_train, Y_train)

Epoch 1, Loss: 0.12374137390188797
Epoch 2, Loss: 0.10830920626279031
Epoch 3, Loss: 0.10651162957982399
Epoch 4, Loss: 0.1055639401719025
Epoch 5, Loss: 0.10432728139206818
Epoch 6, Loss: 0.10363117497935637
Epoch 7, Loss: 0.10301657282858578
Epoch 8, Loss: 0.10288743041888032
Epoch 9, Loss: 0.1025097792303196
Epoch 10, Loss: 0.10187808417622954
Epoch 11, Loss: 0.10217431976526671
Epoch 12, Loss: 0.10155376388758115
Epoch 13, Loss: 0.10166885701462677
Epoch 14, Loss: 0.10148741198074289
Epoch 15, Loss: 0.10121108911143228


In [58]:
def evaluate(model, X, Y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        correct = 0
        total = 0
        for j in range(len(output_cols)):
            _, predicted = torch.max(outputs[j].data, 1)
            correct += (predicted == Y[:, j]).sum().item()
            total += Y.size(0)
        accuracy = correct / total
        print(f'Accuracy: {accuracy:.4f}')

evaluate(model, X_test, Y_test)

Accuracy: 0.5440


In [59]:
def evaluate(model, X, Y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        total_acc = 0
        feature_acc = []

        for j in range(len(output_cols)):
            _, predicted = torch.max(outputs[j].data, 1)
            correct = (predicted == Y[:, j]).sum().item()
            acc = correct / Y.size(0)
            feature_acc.append(acc)
            total_acc += acc

        avg_acc = total_acc / len(output_cols)
        print(f'Average Accuracy: {avg_acc:.4f}')
        for col, acc in zip(output_cols, feature_acc):
            print(f'{col}: {acc:.4f}')

evaluate(model, X_test, Y_test)

Average Accuracy: 0.5440
second_partner_gender: 0.8297
second_partner_age_bin: 0.5884
second_partner_school_category: 0.4995
second_partner_level_id: 0.4361
second_partner_field: 0.3663


### Sample Test - The easiest one to goo...

In [60]:
def predict_partner(sample_row, model, label_encoders):
    """
    Predict partner attributes from a sample input row

    Args:
        sample_row: List of [gender, age, school_category, level_id, field]
        model: Trained TabTransformer model
        label_encoders: Dictionary of fitted LabelEncoders

    Returns:
        Dictionary of predicted attributes
    """
    # Ensure the model is in evaluation mode
    model.eval()

    # Step 1: Prepare input with all required features
    # Now expecting all 5 features in the input
    if len(sample_row) < 5:
        raise ValueError("Input row should contain all 5 features: [gender, age, school_category, level_id, field]")

    full_sample = sample_row  # Use all provided features

    # Step 2: Encode each feature using the corresponding LabelEncoder
    input_cols = [
        'first_partner_gender',
        'first_partner_age_bin',
        'first_partner_school_category',
        'first_partner_level_id',
        'first_partner_field'
    ]

    try:
        encoded_sample = []
        for col, value in zip(input_cols, full_sample):
            # Convert to string and handle missing/unknown values
            value_str = str(value).strip() if value is not None else 'Unknown'

            # Check if the value exists in the encoder's classes
            if value_str not in label_encoders[col].classes_:
                # Find the most common class to use as default
                default_value = label_encoders[col].classes_[0]  # or use specific defaults
                print(f"Warning: Unknown value '{value_str}' for {col}, using '{default_value}' instead")
                value_str = default_value

            encoded_val = label_encoders[col].transform([value_str])[0]
            encoded_sample.append(encoded_val)

    except Exception as e:
        print(f"Encoding error: {e}")
        return None

    # Step 3: Convert to tensor and add batch dimension
    sample_tensor = torch.tensor([encoded_sample], dtype=torch.long)

    # Step 4: Get predictions
    with torch.no_grad():
        output_preds = model(sample_tensor)

    # Step 5: Decode predictions
    output_cols = [
        'second_partner_gender',
        'second_partner_age_bin',
        'second_partner_school_category',
        'second_partner_level_id',
        'second_partner_field'
    ]

    predictions = {}
    for i, col in enumerate(output_cols):
        pred_class = torch.argmax(output_preds[i], dim=1).item()
        predictions[col] = label_encoders[col].inverse_transform([pred_class])[0]

    return predictions

# Example usage with all 5 features
sample_row = [
    'Female',
    '30-34',
    'Ivy League',
    'S4',
    'Business and Financial Occupations'
]

predictions = predict_partner(sample_row, model, label_encoders)

if predictions:
    print("\n=== Predicted Partner Profile ===")
    print(f"Gender:         {predictions['second_partner_gender']}")
    print(f"Age Group:      {predictions['second_partner_age_bin']}")
    print(f"School Category: {predictions['second_partner_school_category']}")
    print(f"Education Level: {predictions['second_partner_level_id']}")
    print(f"Field of Study:  {predictions['second_partner_field']}")


=== Predicted Partner Profile ===
Gender:         Male
Age Group:      30-34
School Category: Other Colleges
Education Level: S5
Field of Study:  Business and Financial Occupations


## Baseline Model

- TODO (?): Compare with a simple MLP
- More for report purpose
  
